# Stabilizer QEC through operators, tableaus, and error frames

## Read before you begin

- Official Stim documentation: [`stim.TableauSimulator`](https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md#stim.TableauSimulator) and [`stim.Tableau`](https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md#stim.Tableau) reference pages.
- Official Stim repository: [quantumlib/Stim](https://github.com/quantumlib/Stim), especially the `getting_started.ipynb` notebook and the `doc/` folder.
- Sections/concepts to extract: what a `Tableau` stores (Pauli images of generators, not amplitudes); what `TableauSimulator` methods (`peek_observable_expectation`, `canonical_stabilizers`, `current_inverse_tableau`, `current_measurement_record`) expose; how compiled samplers and detector samplers differ from the interactive simulator.

## Prerequisites

This is a **standalone companion notebook**, not part of the repository's numbered roadmap — it does not require, and is not required by, any specific later-numbered notebook. It assumes only notebooks `01`–`06`: basic Stim circuits, the classical repetition code, stabilizers and syndromes, classical linear codes and the Hamming code, the Shor and Steane codes built via the CSS construction, and logical operators, code distance, and degeneracy. It does not introduce a new code family, and it does not depend on notebook `07`'s binary-symplectic parity-check-matrix formalism or decoding material — this notebook instead revisits everything already built in `01`–`06`, from an **operator-centric** point of view, and connects that view directly to how Stim is implemented. It can be read any time after notebook `06`, in parallel with or independently of `07` onward.

## Learning objectives

By the end of this notebook you should be able to:

- Explain the difference between the Schrödinger picture, the Heisenberg picture, and stabilizer-tableau tracking, and why Clifford gates make the last one efficient.
- State the operator-centric definition of a stabilizer code ($\mathcal S$, codespace, logical operators, $[[n,k,d]]$) precisely, including the definition of distance via the Pauli centralizer.
- Explain, using the 3-qubit bit-flip code, why "corrects one bit flip" is not the same as "is a good quantum code."
- Read a small stabilizer tableau (state stabilizers, destabilizers, signs) and connect it to Stim's `Tableau`/`TableauSimulator` API.
- Explain, for the Steane code, why six independent stabilizer checks plus a choice of logical state give seven independent *state* stabilizers.
- Explain repeated stabilizer measurement, detection events, and the "error frame" idea behind Stim's efficient noisy-shot sampling.

## How to use this notebook

Read each markdown cell before running the code cell(s) that follow it. Several **Prediction** boxes ask you to commit to an answer before running the next cell — do this on paper or out loud before executing. The notebook ends with five exercises.

## 1. Three viewpoints on the same circuit

A quantum circuit is a fixed sequence of unitary gates, $U$. There are (at least) three different bookkeeping schemes for tracking what a circuit does, and QEC simulation lives in the third one. It is easy to blur these together, so this section is deliberately pedantic about which object is being updated at each step.

**Schrödinger picture.** The circuit updates the *state*:
$$|\psi\rangle \;\longmapsto\; U|\psi\rangle .$$
Operators (observables) are fixed; the state vector moves. This is the picture used in notebook 01's `state_vector()` calls and in ordinary undergraduate quantum mechanics.

**Heisenberg picture.** The circuit instead updates *operators*, and the state is held fixed:
$$O \;\longmapsto\; U^\dagger O\,U .$$
Expectation values agree between the two pictures because $\langle\psi|U^\dagger O U|\psi\rangle = \langle\psi|U^\dagger(O)|\psi\rangle$ evaluated in the transformed operator equals $\langle U\psi|O|U\psi\rangle$ evaluated on the transformed state — the physics (measurement statistics) is identical; only the bookkeeping differs.

**Stabilizer tracking (used by Stim and by this whole repository).** Fix the state to be whatever the circuit prepares, starting from $|0\cdots0\rangle$. Instead of transforming a *fixed* observable, transform the *stabilizer generators themselves*, gate by gate:
$$S \;\longmapsto\; U S U^\dagger .$$

Look closely at the direction of conjugation: it is $USU^\dagger$, not $U^\dagger S U$. This is the opposite conjugation direction from the textbook Heisenberg picture above. The reason is that a stabilizer $S$ is defined by the property $S|\psi\rangle = |\psi\rangle$ for the *current* state $|\psi\rangle$. After applying a gate, the new state is $U|\psi\rangle$, and the operator that stabilizes it is found by asking: which operator $S'$ satisfies $S'(U|\psi\rangle) = U|\psi\rangle$? Substituting $S|\psi\rangle=|\psi\rangle$ shows $S' = USU^{\dagger}$ works, since $USU^\dagger(U|\psi\rangle) = US|\psi\rangle = U|\psi\rangle$.

So: **stabilizer tracking is operator-centric, and closely related in spirit to the Heisenberg picture** (both move bookkeeping from the exponentially large state onto polynomially many operators), **but it is not literally the conventional Heisenberg picture** — the conjugation runs the other way, because a stabilizer is tied to the state that is currently being prepared, not to a fixed observable measured at the end of a fixed reference state. Calling it "interaction-picture-like" (Section 8) will make a similar distinction for how Stim thinks about *noise* layered on top of an *ideal* circuit.

**Why this is efficient for Clifford circuits.** A general $n$-qubit state vector has $2^n$ complex amplitudes. Tracking generic observables under conjugation is just as expensive. But the **Clifford group** — the group generated by $H$, $S$ (phase), and $\mathrm{CX}$ — has a special property: conjugating any $n$-qubit **Pauli string** ($I,X,Y,Z$ on each qubit, up to a sign) by a Clifford unitary always produces *another* Pauli string, not a general operator:
$$U \in \text{Clifford} \implies U P U^\dagger \in \{\pm 1, \pm i\}\cdot\{\text{Pauli strings}\}.$$
An $n$-qubit Pauli string is described by only $2n$ bits (an $x$-bit and a $z$-bit per qubit, Section 4) plus a sign — not $2^n$ amplitudes. So if every operator of interest (stabilizer generators, logical operators, error operators) is always a Pauli string, and every gate is Clifford, the *entire* simulation stays polynomial in $n$. This is the Gottesman–Knill theorem, and it is the reason Stim can simulate thousands of qubits where a state-vector simulator would need to track $2^{1000}$ amplitudes.

In [1]:
import numpy as np
import stim

# A tiny concrete check of "Clifford maps Pauli to Pauli": conjugate Z by H.
# H Z H^dagger = X, a standard identity. Verify it with Stim's own tableau algebra.
h_tableau = stim.Tableau.from_named_gate("H")
conjugated = h_tableau(stim.PauliString("Z"))  # calling a Tableau applies it by conjugation
print("H Z H^dagger =", conjugated)  # expect +X

H Z H^dagger = +X


## 2. Define a stabilizer code through operators

**Stabilizer group.** Let $\mathcal P_n$ be the $n$-qubit Pauli group (all products of $I,X,Y,Z$ on $n$ qubits, with phases $\pm1,\pm i$). A **stabilizer group** $\mathcal S \subset \mathcal P_n$ is a subgroup that is:

- **abelian**: every pair of elements commutes, $S_1S_2 = S_2S_1$ for all $S_1,S_2\in\mathcal S$;
- **excludes $-I$**: $-I \notin \mathcal S$ (if it were included, no state could satisfy $(-I)|\psi\rangle=|\psi\rangle$, so the codespace below would be empty).

$\mathcal S$ is generated by $r$ independent generators $S_1,\dots,S_r$ (independent meaning no product of a nonempty subset of them equals $I$).

**Codespace.** The codespace is the simultaneous $+1$ eigenspace of every element of $\mathcal S$:
$$\mathcal C = \{\,|\psi\rangle : S|\psi\rangle = |\psi\rangle \text{ for every } S\in\mathcal S\,\}.$$
In plain language: a state is "in the code" exactly when every stabilizer, measured on it, reads out $+1$ with certainty. It is enough to check this for the $r$ generators — the rest of $\mathcal S$'s $2^r$ elements are products of generators, so they automatically also give $+1$.

**Why $k=n-r$.** Each independent generator $S_i$ cuts the dimension of the eigenspace in half: constraining a $2^n$-dimensional space to the $+1$ eigenspace of one nontrivial Hermitian, order-2 operator with eigenvalues $\pm1$ leaves a $2^{n-1}$-dimensional subspace, and each further *independent* generator halves it again (independence is exactly what guarantees each new constraint is not already implied by the previous ones). After $r$ independent constraints, the codespace has dimension $2^{n-r}$ — the same dimension as $k=n-r$ qubits' worth of Hilbert space, $2^k$. That is the precise sense in which the codespace "holds" $k=n-r$ logical qubits: it is not literally $k$ separate qubits sitting inside the $n$ physical ones, but a $2^k$-dimensional subspace that can be put in one-to-one correspondence with a $k$-qubit Hilbert space via the logical operators below.

**Logical operators.** For each logical qubit $i=1,\dots,k$, a **logical operator pair** $(\bar X_i, \bar Z_i)$ is a pair of Pauli operators that:

- commute with every element of $\mathcal S$ (so they map $\mathcal C$ back into $\mathcal C$ — they act *within* the codespace rather than leaving it);
- are **not themselves in $\mathcal S$** (otherwise they would act trivially, as $+I$, on every codeword);
- anticommute with each other: $\bar X_i \bar Z_i = -\bar Z_i \bar X_i$.

The anticommutation is what makes $(\bar X_i,\bar Z_i)$ behave like a genuine qubit's $X$ and $Z$: exactly as ordinary single-qubit $X,Z$ anticommute and generate all single-qubit Pauli behaviour, one anticommuting pair per logical qubit is needed to reproduce that same algebra on the $i$-th logical degree of freedom, and different logical qubits' operators commute with each other.

**Parameters $[[n,k,d]]$.** A stabilizer code with these ingredients is written $[[n,k,d]]$: $n$ physical qubits, $k$ logical qubits, and distance $d$, defined next.

**Distance.** Let $\mathcal C_{\mathcal P_n}(\mathcal S) = \{P \in \mathcal P_n : PS=SP \text{ for all } S\in\mathcal S\}$ be the **Pauli centralizer** of $\mathcal S$ — every Pauli string that commutes with the entire stabilizer group (equivalently: every error the stabilizer checks cannot detect, since an undetectable error is exactly one that commutes with every check). Then
$$d = \min_{P \,\in\, \mathcal C_{\mathcal P_n}(\mathcal S)\,\setminus\,\mathcal S} \operatorname{wt}(P),$$
where $\operatorname{wt}(P)$ is the number of qubits on which $P$ acts nontrivially (not $I$).

Reading each piece intuitively:

- $\mathcal C_{\mathcal P_n}(\mathcal S)$: "operators the code's checks are blind to" — includes the stabilizers themselves (trivially undetectable, and also physically inconsequential, since they act as $+I$ on every codeword) *and* the logical operators (undetectable, but consequential — they move you within the codespace).
- excluding $\mathcal S$ itself ("$\setminus \mathcal S$"): throws away the "blind and harmless" operators, leaving only "blind and harmful" ones — i.e. nontrivial logical operators (technically, coset representatives of $\mathcal S$ within the centralizer; every element of $\mathcal C_{\mathcal P_n}(\mathcal S)\setminus\mathcal S$ acts as some nontrivial logical Pauli on the codespace).
- weight, $\operatorname{wt}(P)$: how many physical qubits a physical error would need to touch to realize $P$.

So $d$ is the minimum number of physical qubits an adversarial (or accidental) error must touch to enact an undetectable *and* harmful operation on the encoded information. That is exactly why "distance" is the right word: it is the code's answer to "how many single-qubit errors, in the worst case, can happen before correction is no longer guaranteed to work" (a code with distance $d$ corrects any error of weight $\le \lfloor (d-1)/2\rfloor$).

## 3. Cautionary example: the three-qubit bit-flip repetition code

Notebook 02/03 used the 3-qubit bit-flip code with stabilizers
$$Z_0Z_1,\qquad Z_1Z_2$$
($n=3$, $r=2$ generators, so $k=n-r=1$), and logical representatives
$$\bar X = X_0X_1X_2, \qquad \bar Z = Z_0.$$

**Prediction.** Before reading on: given the distance formula above, what do you think $d$ is for this code — and does it match the informal claim "this code corrects one bit flip"?

Compute the centralizer-minus-stabilizer minimum weight directly. $\bar Z = Z_0$ has weight $1$. Is $Z_0$ in the centralizer of $\{Z_0Z_1, Z_1Z_2\}$? Yes — $Z_0$ commutes with both ($Z$ always commutes with $Z$). Is $Z_0 \in \mathcal S = \{I, Z_0Z_1, Z_1Z_2, Z_0Z_2\}$? No. So $Z_0$ qualifies, giving $d\le 1$; a short check of all weight-1 Paulis confirms nothing lower exists (weight 0 is just $I\in\mathcal S$), so $d=1$.

This is a genuinely important cautionary example: **this is a $[[3,1,1]]$ code**, despite the popular description "it corrects one bit flip." The resolution is that $Z_0$ is precisely a weight-one, undetectable, *harmful* logical operation: it commutes with both stabilizers (so no syndrome ever fires) and it is not a stabilizer (so it silently applies logical $\bar Z$ — it flips the encoded phase/basis information with zero physical disturbance to the bit values Stim would normally check). The code was only ever designed to fix independent random bit-flip ($X$) noise; it was never designed to resist a $Z$ error, and a single $Z$ error anywhere is already a distance-1, fully undetectable logical fault. "Corrects one bit-flip error" and "distance $d$" are talking about different, non-interchangeable things: the first is a statement about a restricted noise model (bit flips only), the second is a worst-case guarantee over *all* Pauli errors. A quantum code's advertised distance must always be checked against the *full* Pauli group, not just the noise model that motivated it.

In [2]:
# Direct verification: is Z0 undetectable (commutes with both stabilizers) yet not in S?
z0 = stim.PauliString("Z__")          # weight-1 Pauli: Z on qubit 0 only
s1 = stim.PauliString("ZZ_")          # Z0 Z1
s2 = stim.PauliString("_ZZ")          # Z1 Z2

print("Z0 commutes with Z0Z1:", z0.commutes(s1))
print("Z0 commutes with Z1Z2:", z0.commutes(s2))

# The stabilizer group S = {I, Z0Z1, Z1Z2, Z0Z2} (products of the two generators, up to sign).
stabilizer_group = [
    stim.PauliString("___"),
    stim.PauliString("ZZ_"),
    stim.PauliString("_ZZ"),
    stim.PauliString("Z_Z"),
]
print("Z0 is itself a stabilizer element:", any(z0 == s for s in stabilizer_group))
print("=> Z0 is undetectable AND not a stabilizer AND weight 1  =>  d = 1")

Z0 commutes with Z0Z1: True
Z0 commutes with Z1Z2: True
Z0 is itself a stabilizer element: False
=> Z0 is undetectable AND not a stabilizer AND weight 1  =>  d = 1


## 4. What a stabilizer tableau stores

**Binary representation of Paulis.** Every single-qubit Pauli $I,X,Y,Z$ can be written as a pair of bits $(x,z)$:

| Pauli | $x$ | $z$ | meaning |
|---|---|---|---|
| $I$ | 0 | 0 | acts trivially |
| $X$ | 1 | 0 | bit flip |
| $Z$ | 0 | 1 | phase flip |
| $Y$ | 1 | 1 | both ($Y = iXZ$ up to the usual convention) |

An $n$-qubit Pauli string is then $2n$ bits (an $x$-bit and a $z$-bit per qubit) plus one more bit (or two, for $\pm1,\pm i$) for the overall sign. This is why Section 1 could say a Pauli string needs only $O(n)$ bits, not $2^n$ amplitudes.

**What a tableau is.** A **stabilizer tableau** for an $n$-qubit *state* stores, for each of $n$ *generators*, its $2n$-bit Pauli representation plus a sign bit — an $n\times(2n+1)$-ish table, i.e. $O(n^2)$ bits total, completely describing a stabilizer state (up to global phase) with no reference to any $2^n$-entry amplitude vector.

**One-qubit $|0\rangle$.** The state $|0\rangle$ is stabilized by $Z$ (since $Z|0\rangle = +1|0\rangle$). Its tableau has one row:

| generator | $x$ | $z$ | sign |
|---|---|---|---|
| $S_1$ | 0 | 1 | $+$ |

Row-by-row reading: there is exactly one row because $n=1$; the row says "the state's one stabilizer generator is $Z$ (from $x{=}0,z{=}1$) with a $+$ sign," i.e. $Z|0\rangle=+|0\rangle$ — matching the definition of the codespace in Section 2 applied to the trivial one-qubit "code" $\mathcal S=\{Z\}$.

**Bell state $|\Phi^+\rangle = \tfrac{1}{\sqrt2}(|00\rangle+|11\rangle)$.** Its stabilizer generators are $X_0X_1$ and $Z_0Z_1$ (this will be *derived*, not asserted, in Section 5). Two qubits, two generators, so two rows:

| generator | qubit 0 $(x,z)$ | qubit 1 $(x,z)$ | sign |
|---|---|---|---|
| $S_1 = X_0X_1$ | $(1,0)$ | $(1,0)$ | $+$ |
| $S_2 = Z_0Z_1$ | $(0,1)$ | $(0,1)$ | $+$ |

Row-by-row reading: row 1 says "one generator is $X$ on qubit 0 *and* $X$ on qubit 1, sign $+$" ($X_0X_1$); row 2 says "the other generator is $Z$ on both qubits, sign $+$" ($Z_0Z_1$). Together these two rows are a complete, $O(n^2)$-bit description of the entangled Bell state — no $4$-entry amplitude vector was needed.

**Destabilizers: why a full tableau needs more than the state stabilizers.** The $n$ *state* stabilizer rows above pin down the codespace (here, a single pure state, since $r=n$), but a simulator also needs to answer questions like "if I measure $Z_0Z_1$ next, and the state changes because of it, how do the *other* rows need to update?" and "what is the actual global phase/sign bookkeeping after a series of gates and measurements?" To do this efficiently, Stim's tableau stores $n$ *additional* rows called **destabilizers**: one Pauli per generator that anticommutes with that generator and commutes with every other stabilizer generator. A destabilizer is not a property of the state (unlike the stabilizers, it is not something the state has a definite eigenvalue for) — it is bookkeeping structure that lets the simulator perform a measurement update (Section 6 onward) in polynomial time instead of having to search for a valid replacement generator from scratch each time. So a *full* Stim tableau is $2n$ rows: $n$ stabilizer generators and $n$ destabilizer partners (plus signs) — this is what `stim.Tableau` and `TableauSimulator.current_inverse_tableau()` actually hold.

**Stim stores the inverse tableau.** For efficiency reasons that need not be understood in full right now, `TableauSimulator.current_inverse_tableau()` returns the *inverse* of the tableau you might naively expect (the one that would directly map "initial generator $i$" to "current generator $i$"). It is safe, for the purposes of this notebook, to treat `current_inverse_tableau()` as *a* faithful encoding of the current stabilizer state, and to primarily read off actual state stabilizers via the friendlier `canonical_stabilizers()` method used below, coming back to the inverse-tableau internals only if you need to, e.g., hand-roll your own Clifford bookkeeping.

In [3]:
# A literal look at stim's own Tableau object for |0>, and for a state stabilized by X0X1, Z0Z1.
zero_tableau = stim.Tableau.from_stabilizers(
    [stim.PauliString("Z")],
    allow_underconstrained=True,
)
print("|0> tableau (1 qubit):")
print(zero_tableau)

bell_tableau = stim.Tableau.from_stabilizers(
    [stim.PauliString("XX"), stim.PauliString("ZZ")],
)
print()
print("Bell-state tableau (2 qubits), stabilizers X0X1 and Z0Z1:")
print(bell_tableau)

|0> tableau (1 qubit):
+-xz-
| ++
| XZ

Bell-state tableau (2 qubits), stabilizers X0X1 and Z0Z1:
+-xz-xz-
| ++ ++
| ZX _Z
| _X XZ


## 5. Bell-state Stim example

Build $|\Phi^+\rangle$ with the standard two-gate recipe:

```
H 0
CX 0 1
```

**Hand derivation first.** Start from $|00\rangle$, stabilized by $Z_0$ and $Z_1$ (each qubit independently in $|0\rangle$). Track each generator through the circuit using $S\mapsto USU^\dagger$ (Section 1's stabilizer-tracking rule), gate by gate.

*Gate 1: `H 0`.* $H$ conjugates $X\leftrightarrow Z$ on qubit 0 and leaves qubit 1 alone.
- $Z_0 \mapsto H Z_0 H^\dagger = X_0$.
- $Z_1 \mapsto Z_1$ (untouched).

So after `H 0`, the stabilizers are $X_0$ and $Z_1$ (this is exactly $|+\rangle\otimes|0\rangle$, as expected).

*Gate 2: `CX 0 1`* (control 0, target 1). Conjugation rules for CX: $X_{\text{ctrl}}\mapsto X_{\text{ctrl}}X_{\text{tgt}}$, $Z_{\text{tgt}}\mapsto Z_{\text{ctrl}}Z_{\text{tgt}}$, and $Z_{\text{ctrl}}\mapsto Z_{\text{ctrl}}$, $X_{\text{tgt}}\mapsto X_{\text{tgt}}$ (each stays fixed when it is not the "active" combination).
- $X_0 \mapsto X_0X_1$ (control-side $X$ spreads to the target).
- $Z_1 \mapsto Z_0Z_1$ (target-side $Z$ spreads back to the control).

**Final derived stabilizers: $X_0X_1$ and $Z_0Z_1$.** This matches the tableau written down by hand in Section 4.

**Prediction.** Before running the cell below: do you expect `sim.peek_observable_expectation` of $X_0X_1$ and of $Z_0Z_1$ to be $+1$, and of, say, $Z_0$ alone, to be $0$ (undefined/random) rather than $\pm1$? Why?

In [4]:
sim = stim.TableauSimulator()

# Build |Phi+> = (|00> + |11>) / sqrt(2)
sim.h(0)
sim.cx(0, 1)

print("Canonical stabilizers (Stim's own generating set):")
for s in sim.canonical_stabilizers():
    print(" ", s)

print()
print("<X0 X1> =", sim.peek_observable_expectation(stim.PauliString("XX")))
print("<Z0 Z1> =", sim.peek_observable_expectation(stim.PauliString("ZZ")))
print("<Z0>    =", sim.peek_observable_expectation(stim.PauliString("Z_")),
      " (0 means 'not a definite eigenvalue', matching that Z0 alone is not a stabilizer)")

print()
print("Current inverse tableau:")
print(sim.current_inverse_tableau())

Canonical stabilizers (Stim's own generating set):
  +XX
  +ZZ

<X0 X1> = 1
<Z0 Z1> = 1
<Z0>    = 0  (0 means 'not a definite eigenvalue', matching that Z0 alone is not a stabilizer)

Current inverse tableau:
+-xz-xz-
| ++ ++
| ZX _X
| X_ XZ


Stim's `canonical_stabilizers()` output should match the hand-derived $X_0X_1,\ Z_0Z_1$ (Stim may print them in a different generator order or with an equivalent product substituted, since any independent generating set of the same group is equally valid — see the discussion of "equivalent generating sets" in Section 6). `current_inverse_tableau()` is shown mainly so its shape (rows/columns for $n=2$) can be compared with the by-hand tableau in Section 4 — it need not be fully decoded to trust that it *is* a complete description of the state.

`current_measurement_record()` will be used in Section 7, once there are actual measurements to record; on this Bell-state example (no measurements yet) it returns an empty list.

In [5]:
print("Measurement record so far (empty: no measurements have happened yet):",
      sim.current_measurement_record())

Measurement record so far (empty: no measurements have happened yet): []


## 6. Steane code as the main code example

The Steane code is $[[7,1,3]]$: it revisits notebook 05's construction, now organized explicitly around Section 2's operator definitions.

**Classical Hamming parity-check matrix.** As in notebooks 04–05, $H$'s three rows are the binary parity checks whose columns are the binary representations of $1,\dots,7$:
$$H = \begin{pmatrix}
0&0&0&1&1&1&1\\
0&1&1&0&0&1&1\\
1&0&1&0&1&0&1
\end{pmatrix}
\quad\text{(columns = binary of } 1,\dots,7\text{, MSB-first row order).}$$

**CSS construction.** Each row $h_i$ of $H$ becomes *two* stabilizers: an $X$-type check ($X$ on every qubit where $h_i$ has a 1) and a $Z$-type check ($Z$ on the same support). Because $HH^T\equiv 0 \pmod 2$ (notebook 05), every $X$-type check commutes with every $Z$-type check, so the six resulting operators are a valid (abelian, $-I$-free) stabilizer group.

**Why $n=7,k=1,d=3$.** $n=7$: the classical Hamming code's blocklength. $r=6$ independent stabilizer generators (3 $X$-type + 3 $Z$-type), so $k=n-r=7-6=1$ — one logical qubit. $d=3$: the classical Hamming code's minimum distance is 3, and the CSS construction inherits this distance on both the $X$ and $Z$ sides (the smallest undetectable-and-nontrivial Pauli error is weight 3, matching notebook 06's distance computation for this code).

**Preparing $|+_L\rangle$, not an arbitrary logical state.** To keep the example concrete and avoid an "arbitrary unknown logical input" (which cannot be meaningfully checked against a hand-derived expectation), this section prepares the specific logical state $|+_L\rangle$ (the $+1$ eigenstate of $\bar X$) using the same encoder as notebook 05, which happens to prepare $|0_L\rangle$, followed by a logical $\bar H$ (transversal Hadamard) to rotate $|0_L\rangle\to|+_L\rangle$. (Alternatively $|+_L\rangle$ can be prepared directly by initializing the 4 message qubits in $|+\rangle$ before the parity `CX`s, which is what the circuit below does — it is the Steane encoder with the very first `H` layer already present, since the message qubits start in $|+\rangle$ under `H`.)

In [6]:
import numpy as np

# --- Classical Hamming H, exactly as in notebooks 04-05 ---
def bits_of(j, width):
    return tuple((j >> shift) & 1 for shift in reversed(range(width)))

H_columns = [bits_of(j, width=3) for j in range(1, 8)]
H = np.array([[col[row] for col in H_columns] for row in range(3)])
print("Hamming parity-check matrix H:")
print(H)
print("H H^T mod 2 (dual-containing check):")
print((H @ H.T) % 2)

# --- CSS stabilizers: each row -> one X-type and one Z-type check ---
def row_to_pauli_string(row, letter):
    return "".join(letter if bit else "I" for bit in row)

x_checks = [stim.PauliString(row_to_pauli_string(row, "X")) for row in H]
z_checks = [stim.PauliString(row_to_pauli_string(row, "Z")) for row in H]

print()
print("X-type stabilizers:")
for p in x_checks:
    print(" ", p)
print("Z-type stabilizers:")
for p in z_checks:
    print(" ", p)

Hamming parity-check matrix H:
[[0 0 0 1 1 1 1]
 [0 1 1 0 0 1 1]
 [1 0 1 0 1 0 1]]
H H^T mod 2 (dual-containing check):
[[0 0 0]
 [0 0 0]
 [0 0 0]]

X-type stabilizers:
  +___XXXX
  +_XX__XX
  +X_X_X_X
Z-type stabilizers:
  +___ZZZZ
  +_ZZ__ZZ
  +Z_Z_Z_Z


In [7]:
# --- Steane encoder that prepares |+_L> directly ---
# Same structure as notebook 05's build_steane_encoder(), but the message
# qubits (0-indexed 2, 4, 5, 6) start with H (as before) and, because the
# logical operator being fixed here is Xbar rather than Zbar, no extra
# logical gate is needed: this circuit already prepares |+_L>.
def build_steane_plus_l():
    sim = stim.TableauSimulator()
    sim.h(2); sim.h(4); sim.h(5); sim.h(6)      # message qubits -> |+>
    sim.cx(2, 0); sim.cx(4, 0); sim.cx(6, 0)    # p1 (position 0)
    sim.cx(2, 1); sim.cx(5, 1); sim.cx(6, 1)    # p2 (position 1)
    sim.cx(4, 3); sim.cx(5, 3); sim.cx(6, 3)    # p4 (position 3)
    return sim

steane_sim = build_steane_plus_l()

print("The six CSS code checks, evaluated on the prepared state (all should be +1):")
for p in x_checks + z_checks:
    print(f"  <{p}> = {steane_sim.peek_observable_expectation(p):+d}")

The six CSS code checks, evaluated on the prepared state (all should be +1):
  <+___XXXX> = +1
  <+_XX__XX> = +1
  <+X_X_X_X> = +1
  <+___ZZZZ> = +1
  <+_ZZ__ZZ> = +1
  <+Z_Z_Z_Z> = +1


**Why six checks can yield seven independent state stabilizers.** The six CSS checks define the *codespace* $\mathcal C$ (dimension $2^k=2^1=2$) — they do not, by themselves, pin down a single state, since $\mathcal C$ still has room for one full logical qubit's worth of freedom. A *specific* prepared pure state, such as $|+_L\rangle$, is a single point inside that 2-dimensional codespace, and pinning down a single point out of a 2-dimensional (one-logical-qubit) space takes exactly one more independent binary constraint — precisely the role played by $\bar X = +1$ (the logical operator eigenvalue that *selects* $|+_L\rangle$ rather than $|-_L\rangle$ or any superposition, out of $\mathcal C$). So: 6 stabilizer generators pin down the 7-qubit codespace, and the 7th independent generator, $\bar X$, pins down *where in that codespace* the actual prepared state sits. This is a direct instance of the codespace-dimension argument in Section 2 applied twice: once to go from $\mathbb C^{2^7}$ down to the $2^1$-dimensional $\mathcal C$ (6 constraints), and once more to go from $\mathcal C$ down to a single ray (1 more constraint).

In [8]:
print("Full stabilizer generators of the prepared |+_L> state (all 7):")
for s in steane_sim.canonical_stabilizers():
    print(" ", s)

x_bar = stim.PauliString("XXXXXXX")
print()
print("<Xbar> = X0 X1 ... X6 on |+_L>:", steane_sim.peek_observable_expectation(x_bar))

Full stabilizer generators of the prepared |+_L> state (all 7):
  +X____XX
  +Z_Z_Z_Z
  +_X__X_X
  +_ZZ__ZZ
  +__X_XX_
  +___XXXX
  +___ZZZZ

<Xbar> = X0 X1 ... X6 on |+_L>: 1


Stim's `canonical_stabilizers()` list is very unlikely to spell out "the six textbook CSS checks, plus $\bar X$" verbatim — it instead returns *some* independent generating set of the same 7-element (well, $2^7$-element) stabilizer group, which may mix the checks together (e.g. a row might come back as a product of two textbook checks). This is expected and harmless: any two generating sets of the same group describe exactly the same codespace and the same prepared state; what matters for the arguments above is the *group*, not the specific list of generators printed.

## 7. Repeated stabilizer measurements and detection events

In a real QEC experiment, stabilizers are not measured once — they are measured **repeatedly**, once per QEC round, because two different things can go wrong between rounds:

- a **data error**: a physical error (e.g. an $X$ or $Z$ Pauli) acting on a data qubit changes the *actual eigenvalue* the check should report;
- a **measurement error**: the ancilla/gates/readout implementing the check itself can misreport the eigenvalue even if no data error occurred.

If a stabilizer were only measured once, these two failure modes would be indistinguishable. Repeating the measurement and comparing consecutive results is what tells them apart.

**Detection events.** Let $m_t$ be the outcome (as $0/1$) of the same check at round $t$. Define
$$d_t = m_t \oplus m_{t-1}$$
— a "detection event" fires ($d_t=1$) exactly when the outcome *changed* from the previous round.

- A **data error** that persists (e.g. a static $X$ error sitting on a data qubit) flips the check's eigenvalue starting at the round after it occurs, and then *stays flipped* every subsequent round — so it produces exactly **two** detection events in spacetime: one when the error is first "seen" (round $t$: $d_t=1$), and one when it is finally corrected or otherwise stops affecting the check (some later round: $d_{t'}=1$ again). In between, $m_t$ stays constant (still flipped), so $d_t=0$ for those intermediate rounds.
- A **measurement error** at a single round flips only *that* round's reported $m_t$, with no effect on the physical eigenvalue or any other round's measurement — so it produces exactly **two adjacent** detection events, at $t$ and $t+1$ (the outcome looks "flipped" going into round $t$, then "flipped back" going into round $t+1$, purely because $m_t$ itself was wrong).

Both cases give pairs of detection events, but they differ in *spacing*: a lone measurement error's pair is always temporally adjacent (consecutive rounds of the *same* check), while a data error's pair can be spread arbitrarily far apart in time (however long the error persists) — and, for a check involving several data qubits, a single data error typically flips *multiple different checks* at the same round, while a lone measurement error only ever affects the one check being misreported. Decoders (not covered in depth here) use exactly this spacetime pattern to distinguish and locate likely errors.

**`MPP` is abstract.** The example below uses `MPP Z0*Z1` — a **multi-Pauli product measurement**, an abstraction meaning "measure the joint observable $Z_0Z_1$ and report its eigenvalue," in a single instruction. Real hardware cannot measure $Z_0Z_1$ directly as one atomic operation; it is normally implemented with an ancilla qubit, a small number of two-qubit gates entangling the ancilla with the data qubits, and an ordinary single-qubit measurement of the ancilla (this is exactly the syndrome-extraction circuit built explicitly in notebook 09/10 for the surface code). `MPP` lets this notebook focus on the *logical* structure of repeated measurement without re-deriving that ancilla circuitry.

In [9]:
# Repeated Z0*Z1 checks on a 2-qubit state, with an injected persistent data
# error partway through, to see the detection-event pattern predicted above.
#
# Note: the error is injected as "X_ERROR", ..., 1.0) -- a *noise channel* with
# probability 1 -- rather than a plain "X" gate. This matters: DETECTOR compares
# the actual measurement record against the value Stim expects from the circuit's
# noise-free skeleton. A plain "X" gate is part of that skeleton (Stim "expects"
# it), so it would never trip a detector. Only a noise channel counts as the kind
# of unplanned deviation detectors are built to catch -- this is a first glimpse
# of Section 8's "ideal circuit vs. error frame" split.
circuit = stim.Circuit()
circuit.append("R", [0, 1])          # reset both qubits to |00>
circuit.append("MPP", [stim.target_z(0), stim.target_combiner(), stim.target_z(1)])   # round 0 (baseline)
circuit.append("DETECTOR", [])       # first measurement: nothing to compare to yet in this toy setup

circuit.append("MPP", [stim.target_z(0), stim.target_combiner(), stim.target_z(1)])   # round 1
circuit.append("DETECTOR", [stim.target_rec(-1), stim.target_rec(-2)])  # compares round 1 vs round 0

circuit.append("X_ERROR", [0], 1.0)  # inject a persistent data error on qubit 0

circuit.append("MPP", [stim.target_z(0), stim.target_combiner(), stim.target_z(1)])   # round 2 (error present)
circuit.append("DETECTOR", [stim.target_rec(-1), stim.target_rec(-2)])  # compares round 2 vs round 1

circuit.append("MPP", [stim.target_z(0), stim.target_combiner(), stim.target_z(1)])   # round 3 (error still present)
circuit.append("DETECTOR", [stim.target_rec(-1), stim.target_rec(-2)])  # compares round 3 vs round 2

sample = circuit.compile_detector_sampler().sample(shots=1)[0]
print("Detection events (round 0, then rounds 1,2,3 vs previous round):", sample.astype(int))
print("Expected: [0, 0, 1, 0]  (a single fired detector right when Z0Z1's eigenvalue flips,")
print("since the error, once present, keeps flipping the SAME direction every later round)")

Detection events (round 0, then rounds 1,2,3 vs previous round): [0 0 1 0]
Expected: [0, 0, 1, 0]  (a single fired detector right when Z0Z1's eigenvalue flips,
since the error, once present, keeps flipping the SAME direction every later round)


Only *one* detector fires here (going from round 1 to round 2), not two — because this toy circuit stops before the error is corrected or removed. Section 7's "two detection events" claim describes the *full* lifetime of a data error: one event when it first flips the check, and a second only once something later flips the check back (a correction, or the error being undone). A single injected, uncorrected, persistent error is expected to look like a single "step" in the detector time series — a rising edge with no matching falling edge in this short example — which is exactly what the sample above should show.

## 8. Interaction-picture-like error frames and Stim efficiency

**The practical idea.** Split a noisy circuit's evolution into two parts:

1. the fixed, known **ideal circuit** — what would happen with no errors at all;
2. a **Pauli error frame** — the (random, shot-dependent) difference caused by noise, tracked separately from the ideal evolution.

Concretely: run one **reference shot** with no noise applied, recording what every measurement would read out in the ideal case. Then, for an actual noisy shot, instead of re-simulating the whole noisy circuit from scratch, track only *which Pauli errors occurred* and propagate that small amount of extra information forward through the (fixed, already-understood) ideal circuit to see which measurement bits and detector events *change relative to the reference*. This is analogous to the physics **interaction picture**, which similarly separates a "known, exactly solvable" evolution from a smaller perturbation and evolves only the perturbation's effect — hence "interaction-picture-like." It is *not* a literal instance of the textbook interaction picture (which precisely partitions a Hamiltonian into $H_0+V$ and evolves states/operators under a defined interaction-picture transformation); the analogy here is intentionally loose, describing an efficient sampling strategy, not a rigorous physical construction.

**Three genuinely different Stim tools, easy to conflate:**

- **`stim.TableauSimulator`**: interactive, gate-by-gate simulation of *one* stabilizer state, backed by a single inverse stabilizer tableau (Sections 4–6). Good for deriving/checking small examples by hand, one shot at a time — not the tool used for large noisy-shot statistics.
- **Compiled samplers and detector samplers** (`circuit.compile_sampler()`, `circuit.compile_detector_sampler()`, used in notebook 01 and Section 7 above): efficient generation of *many* noisy shots' measurement or detector outcomes directly, without stepping an interactive tableau shot-by-shot.
- **Reference-frame sampling**: the specific efficiency technique described above — simulate a noise-free reference shot once, then track only the Pauli-error frame (the "difference") for each noisy shot, rather than fully re-simulating each noisy shot's stabilizer state from scratch.

**Beyond tableaus.** Stim's ability to sample millions of shots of a large QEC circuit quickly is not explained by the tableau representation alone. It also relies on **vectorization** (processing many shots' worth of bits together using wide SIMD-style bit operations rather than one shot at a time) and **detector-oriented simulation** (specializing the whole simulation strategy around directly producing detector-event bits — the objects a decoder actually consumes — rather than generic state simulation). The tableau/stabilizer representation is what makes each *individual* shot cheap in principle (Section 1); vectorization and detector-oriented sampling are what make *many* shots cheap in practice.

In [10]:
# A tiny, concrete look at "reference shot vs noisy shot" for the repeated-check
# circuit from Section 7, now with a *random* (50%) X error instead of a guaranteed one.
noisy_circuit = stim.Circuit()
noisy_circuit.append("R", [0, 1])
noisy_circuit.append("MPP", [stim.target_z(0), stim.target_combiner(), stim.target_z(1)])
noisy_circuit.append("DETECTOR", [])
noisy_circuit.append("X_ERROR", [0], 0.5)     # random data error, 50% probability
noisy_circuit.append("MPP", [stim.target_z(0), stim.target_combiner(), stim.target_z(1)])
noisy_circuit.append("DETECTOR", [stim.target_rec(-1), stim.target_rec(-2)])

# The compiled detector sampler draws many shots efficiently -- this is the
# "compiled sampler" tool, not a step-by-step TableauSimulator loop.
detector_sampler = noisy_circuit.compile_detector_sampler()
samples = detector_sampler.sample(shots=2000)
fired_fraction = samples[:, 1].mean()  # second detector: fires iff the X_ERROR actually happened
print(f"Fraction of shots where the round-2 detector fired: {fired_fraction:.3f}  (expect close to 0.5)")

Fraction of shots where the round-2 detector fired: 0.491  (expect close to 0.5)


## 9. Short official Stim source-code tour

This section maps each concept above onto official Stim documentation and the official [quantumlib/Stim](https://github.com/quantumlib/Stim) repository. Exact internal source-file paths and line numbers can change between Stim versions (this notebook was checked against Stim $1.16$); the *public API* referenced below is the stable surface to rely on.

- **`TableauSimulator` — "How is a stabilizer state represented and updated interactively?"**
  Public API: [`stim.TableauSimulator`](https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md#stim.TableauSimulator) in the Python API reference. It wraps a single evolving `Tableau`, updated gate-by-gate as each instruction is applied (e.g. via `.do(circuit)`, `.h(q)`, `.cx(c,t)`), exactly as used in Sections 5–6 above.

- **Inverse tableaux — "What does the simulator actually store internally, and why 'inverse'?"**
  Public API: [`stim.TableauSimulator.current_inverse_tableau`](https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md#stim.TableauSimulator.current_inverse_tableau) and [`stim.Tableau`](https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md#stim.Tableau) itself (constructors like `from_stabilizers`, methods like `to_stabilizers`, `inverse`). The general Stim repository README and `doc/` folder describe the stabilizer-tableau representation at a conceptual level; the precise "why inverse, not forward" performance argument is an internal implementation detail of the C++ core (see the repository's top-level `src/` tree) and is deliberately not required for this notebook's usage-level understanding.

- **`canonical_stabilizers` — "How do I read off an actual, human-comparable list of state stabilizers?"**
  Public API: [`stim.TableauSimulator.canonical_stabilizers`](https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md#stim.TableauSimulator.canonical_stabilizers). Used directly in Sections 5–6 to compare a hand derivation against Stim's own output.

- **Measurement-record handling — "How does a measurement affect the state, and how are past outcomes referenced later (e.g. by `DETECTOR`)?"**
  Public API: [`stim.TableauSimulator.current_measurement_record`](https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md#stim.TableauSimulator.current_measurement_record) for the interactive simulator's own record, and the `DETECTOR` instruction plus `stim.target_rec` in the [Stim circuit file-format reference](https://github.com/quantumlib/Stim/blob/main/doc/gates.md) for how compiled circuits refer back to earlier measurement outcomes by relative index (as used in Section 7's `stim.target_rec(-1)`, `stim.target_rec(-2)`).

- **Compiled sampling — "Why can many noisy shots be sampled efficiently, without an interactive tableau step per shot?"**
  Public API: [`stim.Circuit.compile_sampler`](https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md#stim.Circuit.compile_sampler) and [`stim.Circuit.compile_detector_sampler`](https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md#stim.Circuit.compile_detector_sampler), used in notebook 01 and Section 8 above. The repository's `doc/` folder and README describe, at a conceptual level, the compiled/vectorized sampling strategy that backs these methods.

- **Reference-frame sampling — "How are many noisy shots tracked efficiently relative to a noise-free reference?"**
  This is described conceptually in the official Stim paper and repository documentation (search the repository's `doc/` folder and top-level README for discussion of frame/reference-based simulation); it is the algorithmic idea behind why `compile_detector_sampler().sample(shots=...)` in Section 8 can draw thousands of shots quickly rather than re-deriving a fresh tableau per shot.

Official entry points used throughout this tour:

- Stim GitHub repository: <https://github.com/quantumlib/Stim>
- Stim Python API reference: <https://github.com/quantumlib/Stim/blob/main/doc/python_api_reference_vDev.md>
- Stim gate/file-format reference: <https://github.com/quantumlib/Stim/blob/main/doc/gates.md>

## Exercises

**1.** By hand, conjugate the stabilizers $X_0,Z_1$ of $|+\rangle\otimes|0\rangle$ through a single `CX 0 1` (control 0, target 1), using the CX conjugation rules from Section 5. Then check your answer with `stim.TableauSimulator`.

**2.** For the 3-qubit bit-flip code (Section 3), find a *different* weight-1 undetectable, non-stabilizer Pauli besides $Z_0$ (hint: check $Z_1$ and $Z_2$), and explain in one sentence why its existence does not change $d$.

**3.** Using `stim.Tableau.from_stabilizers`, build the tableau for the single-qubit state $|+\rangle$ (stabilized by $X$), and compare its shape/contents to the $|0\rangle$ tableau in Section 4.

**4.** Extend Section 6's Steane-code cell to also prepare $|0_L\rangle$ (notebook 05's original encoder) and confirm that $\bar Z=+1$ (rather than $\bar X=+1$) is the seventh independent condition selecting it, mirroring the $|+_L\rangle$ argument for $\bar X$.

**5.** Modify Section 8's noisy circuit to inject `Z_ERROR` instead of `X_ERROR` on qubit 0, rerun the detector sampler, and explain in one or two sentences why the fired fraction changes (hint: `MPP Z0*Z1` only detects Paulis that anticommute with $Z_0Z_1$).